# Setup

In [30]:
from pathlib import Path
import os
import subprocess
import shutil
import time
import pandas as pd
import sys
from functools import partial

# Tools
Note: All functions (except d2n) input patient folder, then everything will auto sort.

## D2N
Includes special sorting for dataset5 folder structure. Does not input the usual patient folder path.

Inputs patient number path without the _date. Example: /mnt/1248/open/dataset5/gLymphVIDA/Hydro_1/0662719

Cannot chain run.

In [95]:
def d2n(d_patient_folderpath: str, n_group_folderpath: str, silent = False, **kwargs):
    
    patientid = Path(d_patient_folderpath).name
    input_folderpath = Path(d_patient_folderpath)

    if not silent:
        print(f"Searching {input_folderpath} for MRI folders.")
        print(f"Found {sum(1 for f in Path().iterdir() if f.is_dir())} folders in {patientid}.")

    

    giveup = []

    for folder in input_folderpath.iterdir():
        if Path(f"{folder}/MRI").exists():

            flag_count = 0
            date = folder.name
            output_folderpath = Path(n_group_folderpath) / f"{patientid}_{date}" / "raw"
            !mkdir -p {output_folderpath}

            while not any(output_folderpath.iterdir()): # Check if the folder got the nii files, if not then flag and restart.
                if not silent:
                    print(f"Running dcm2niix for {patientid}_{date}.")
                dcm_folder = folder / "MRI" / "dDTI_b1000_d30"
                if Path(dcm_folder).exists():
                    # cmd = [
                    #     "dcm2niix",
                    #     "-z", "y",
                    #     "-o", str(output_folderpath),
                    #     str(dcm_folder),
                    # ]
                    # subprocess.run(cmd, capture_output=True, text=True, check=True)
                    if len(list(Path(dcm_folder).iterdir())) >= 300:
                        print("Too many files. Skipping.")
                        giveup.append(f"{patientid}_{date}")
                        break

                    # for file in Path(dcm_folder).iterdir():
                    #     print(file.name)
                    

                    !dcm2niix -z y -o {output_folderpath} {dcm_folder}


                    if not any(output_folderpath.iterdir()):
                        flag_count += 1
                        print(f"Output not found. ({flag_count})")
                    if flag_count == 3:
                        print(f"Flagged 3 times, skipping {patientid}_{date}.")
                        giveup.append(f"{patientid}_{date}")
                        break
                    if any("DeIdentified" in f.name for f in Path(output_folderpath).iterdir()):
                        print(f"Removed D2N result for patient \033[4m{patientid}\033[0m because deidentified.")
                        giveup.append(f"{patientid}_{date}")
                        break
                    
                else: 
                    print("Folder format weird.")
                    giveup.append(f"{patientid}_{date}")
                    break

    return {"d2n_empty_folders": giveup}

## BET

In [96]:
FSLDIR = "/mnt/1248/open/usr/local/fsl"
fsl_setup = subprocess.run(
    ["bash", "-c", f"source {FSLDIR}/etc/fslconf/fsl.sh && env"],
    check=True,
    capture_output=True,
    text=True,
)

for line in fsl_setup.stdout.splitlines():
    key, separator, value = line.partition("=")
    if separator:
        os.environ[key] = value

os.environ["FSLDIR"] = FSLDIR
os.environ["PATH"] = f"{FSLDIR}/bin:{os.environ['PATH']}"

print(f"FSLDIR={os.environ['FSLDIR']}")
print(f"fslroi={shutil.which('fslroi')}")

FSLDIR=/mnt/1248/open/usr/local/fsl
fslroi=/mnt/1248/open/usr/local/fsl/bin/fslroi


In [97]:
def bet(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_raw = Path(patient_folderpath) / "raw"
    niigz_files = list(path_raw.glob('*.nii.gz'))

    if Path(f"{folderpath}/bet/{patientid}_b0_R_mask.nii.gz").exists() == True:
        if not silent:
            print(f"Mask for {patientid} already exists. Skipping BET.")
        return
    
    if len(niigz_files) != 1:
        print(f"Warning: Found {len(niigz_files)} .nii.gz files in {path_raw}.")
    if not niigz_files:
        raise FileNotFoundError(f"No .nii.gz file found in {path_raw}")

    niigz_file = niigz_files[0].name
    if not silent:
        print(f"Selected {niigz_file} for mask generation.")

    !mkdir -p {folderpath}/bet

    !fslroi {path_raw}/{niigz_file} {folderpath}/bet/{patientid}_b0.nii.gz 0 1

    !bet {folderpath}/bet/{patientid}_b0.nii.gz {folderpath}/bet/{patientid}_b0_R.nii.gz -m -f 0.3 -R

    !bet {folderpath}/bet/{patientid}_b0.nii.gz {folderpath}/bet/{patientid}_b0_noR.nii.gz -m -f 0.3
    
    # subprocess.run(
    #     [
    #         "bet",
    #         f"{folderpath}/bet/{patientid}_b0.nii.gz",
    #         f"{folderpath}/bet/{patientid}_b0_R.nii.gz",
    #         "-m",
    #         "-f",
    #         "0.3",
    #         "-R",
    #     ],
    #     capture_output=True,
    #     text=True,
    #     check=True,
    # )

    # subprocess.run(
    #     [
    #         "bet",
    #         f"{folderpath}/bet/{patientid}_b0.nii.gz",
    #         f"{folderpath}/bet/{patientid}_b0_noR.nii.gz",
    #         "-m",
    #         "-f",
    #         "0.3",
    #     ],
    #     capture_output=True,
    #     text=True,
    #     check=True,
    # )

    if not silent:
        print(f"Generated masks for {patientid}.")

    !fslmaths {folderpath}/bet/{patientid}_b0_R_mask.nii.gz -sub {folderpath}/bet/{patientid}_b0_noR_mask.nii.gz {folderpath}/bet/diff.nii.gz

    diff = !fslstats {folderpath}/bet/diff.nii.gz -R
    if not silent:
        print("Diff: " + diff[0])
    # if diff[0] != "0.000000 0.000000":
    #     print("The two masks are different. Please check the results.")
    # else:
    #     print("The two masks are the same. Safe to proceed.")

    return None


## EDDY

In [98]:
def eddy(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_raw = Path(patient_folderpath) / "raw"
    niigz_files = list(path_raw.glob('*.nii.gz'))

    if Path(f"{folderpath}/eddy/{patientid}.nii.gz").exists() == True:
        if not silent:
            print(f"EDDY correction for {patientid} already exists. Skipping EDDY.")
        return

    if len(niigz_files) != 1:
        print(f"Warning: Found {len(niigz_files)} .nii.gz files in {path_raw}.")
        # print(niigz_files)
    if not silent:
        print(f"Selected {niigz_files[0].name} for EDDY correction.")
    niigz_file = niigz_files[0].name
    filename_base = niigz_file.replace(".nii.gz", "")
    # print(f"Filename base: {filename_base}")

    !mkdir -p {folderpath}/eddy

    # Make acqparams.txt
    grep = !grep -iE "PhaseEncodingDirection|TotalReadoutTime" "{path_raw}/{filename_base}.json"
    # print(grep)
    trt = grep[0].split(":")[1].strip().strip(",")
    ped = grep[1].split(":")[1].strip().strip("\",")
    
    if ped == "i":
        ped = "1 0 0"
    if ped == "i-":
        ped = "-1 0 0"
    if ped == "j":
        ped = "0 1 0"
    if ped == "j-":
        ped = "0 -1 0"
    if ped == "k":
        ped = "0 0 1"
    if ped == "k-":
        ped = "0 0 -1"

    with open(f"{folderpath}/eddy/acqparams.txt", "w") as f:
        f.write(f"{ped} {trt}")
    if not silent:
        print(f"Created acqparams.txt with {ped} {trt}.")

    # Make index.txt
    n = int(subprocess.check_output(["fslval", f"{path_raw}/{filename_base}.nii.gz", "dim4"], text=True).strip())
    index_line = " ".join(["1"] * n)
    with open(f"{folderpath}/eddy/index.txt", "w") as f:
        f.write(index_line)
    if not silent:
        print(f"Created index.txt with n = {n}.")

    # Run EDDY correction
    if not silent:
        print(f"Running eddy_cuda10.2 for {patientid}...")
    
    subprocess.run(
        [
            "eddy_cuda10.2",
            f"--imain={path_raw}/{filename_base}.nii.gz",
            f"--mask={folderpath}/bet/{patientid}_b0_R_mask.nii.gz",
            f"--acqp={folderpath}/eddy/acqparams.txt",
            f"--index={folderpath}/eddy/index.txt",
            f"--bvecs={path_raw}/{filename_base}.bvec",
            f"--bvals={path_raw}/{filename_base}.bval",
            f"--out={folderpath}/eddy/{patientid}",
            "--repol",
            "--cnr_maps",
            "--data_is_shelled",
            
            # " > /dev/null 2>&1"
        ],
        check = True,
    ) 

    # Copy bval and bvec files to the eddy folder
    bval_file = list(path_raw.glob('*.bval'))[0]
    !cp {bval_file} {folderpath}/eddy/{patientid}.bval

    bvec_file = list(path_raw.glob('*.bvec'))[0]
    !cp {bvec_file} {folderpath}/eddy/{patientid}.bvec

    return None



## SRC

In [99]:
def src(patient_folderpath: str, silent = False, **kwargs) -> None:

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)
    
    path_eddy = Path(patient_folderpath) / "eddy"
    niigz_file = list(path_eddy.glob(f'{patientid}.nii.gz'))[0].name

    if Path(f"{folderpath}/src/{patientid}.sz").exists() == True:
        if not silent:
            print(f"SRC conversion for {patientid} already exists. Skipping SRC.")
        return
    
    if not silent:
        print(f"Selected {niigz_file} for SRC conversion.")

    !mkdir -p {folderpath}/src

    cmd = [
        "dsi_studio",
        "--action=src",
        f"--source={folderpath}/eddy/{patientid}.nii.gz",
        f"--bval={folderpath}/eddy/{patientid}.bval",
        f"--bvec={folderpath}/eddy/{patientid}.bvec",
        f"--output={folderpath}/src/{patientid}",
    ]
    subprocess.run(cmd, capture_output=True, text=True)
    if not silent:
        print(f"Converted {niigz_file} to SRC format.")
    
    return None

## REC

### RECdefault

In [100]:
def rec(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_src = Path(patient_folderpath) / "src"
    src_file = list(path_src.glob(f'{patientid}.sz'))[0].name
    
    if Path(f"{folderpath}/rec/{patientid}.fib.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid} already exists. Skipping REC.")
        return
    
    if not silent:
        print(f"Selected {src_file} for FIB conversion.")

    !mkdir -p {folderpath}/rec

    cmd = [
        "dsi_studio",
        "--action=rec",
        f"--source={folderpath}/src/{patientid}.sz",
        f"--mask={folderpath}/bet/{patientid}_b0_R_mask.nii.gz",
        f"--method=1",
        f"--other_output=fa,md,ad,rd,gfa,qa,rdi,nrdi",
        f"--output={folderpath}/rec/{patientid}.fib.gz",
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Converted {src_file} to FIB format.")

    return None

### RECqsdr


In [101]:
def rec_qsdr(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_src = Path(patient_folderpath) / "src"
    src_file = list(path_src.glob(f'{patientid}.sz'))[0].name
    
    if Path(f"{folderpath}/rec_qsdr/{patientid}.fib.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid} already exists. Skipping REC (QSDR).")
        return
    
    if not silent:
        print(f"Selected {src_file} for FIB conversion.")

    !mkdir -p {folderpath}/rec_qsdr

    cmd = [
        "dsi_studio",
        "--action=rec",
        f"--source={folderpath}/src/{patientid}.sz",
        f"--mask={folderpath}/bet/{patientid}_b0_R_mask.nii.gz",
        f"--method=7",
        f"--param0=1.25",
        f"--param1=2",
        f"--check_btable=1",
        f"--motion_correction=0",
        f"--output={folderpath}/rec_qsdr/{patientid}.fib.gz",
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Converted {src_file} to FIB format.")

    return None

### RECcustom

In [102]:
def rec_custom(patient_folderpath: str, 
               src_foldername: str = "src", 
               rec_foldername: str = "rec_custom", 
               rec_method: int = 1,
               rec_param0: float = 1.25,
               rec_param1: float = 2,
               silent = False, 
               **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_src = Path(patient_folderpath) / src_foldername
    src_file = list(path_src.glob(f'{patientid}.sz'))[0].name


    
    if Path(f"{folderpath}/{rec_foldername}/{patientid}.fib.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid} already exists. Skipping REC.")
        return
    
    if not silent:
        print(f"Selected {src_file} for FIB conversion.")

    !mkdir -p {folderpath}/{rec_foldername}

    cmd = [
        "dsi_studio",
        "--action=rec",
        f"--source={folderpath}/{src_foldername}/{patientid}.sz",
        f"--mask={folderpath}/bet/{patientid}_b0_R_mask.nii.gz",
        f"--method={rec_method}",
        f"--param0={rec_param0}",
        f"--param1={rec_param1}",
        f"--check_btable=1",
        f"--motion_correction=0",
        f"--output={folderpath}/{rec_foldername}/{patientid}.fib.gz",
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Converted {src_file} to FIB format.")

    return None

## TRK

### TRKdefault

In [103]:
def trk(patient_folderpath: str, silent = False, **kwargs):
    
    
    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    
    if Path(f"{folderpath}/trk/{patientid}_{trk_label}.tt.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid}_{trk_label} already exists. Skipping TRK.")
        return

    path_rec = Path(patient_folderpath) / "rec"
    # print(path_rec)
    fib_file = list(path_rec.glob(f'{patientid}.fib.gz'))[0].name
    if not silent:
        print(f"Selected {fib_file} for fiber tracking.")

    roi = kwargs.get("trk_roi", ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior", "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior", "FreeSurferSeg:CC_Anterior"])
    if not silent:
        print(roi)
    !mkdir -p {folderpath}/trk

    cmd = [
        "dsi_studio",
        "--action=trk",
        f"--source={folderpath}/rec/{patientid}.fib.gz",
        f"--seed={','.join(roi)}",
        f"--ter={','.join(roi)},dilation,dilation,negate",
        f"--min_length=5",
        f"--tract_count=600000",
        f"--output={folderpath}/trk/{patientid}_{trk_label}.tt.gz",
        f"--export=stat",
        # " > /dev/null 2>&1"
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Successfully tracked fibers for {patientid}_{trk_label}.")

    return

### TRKcustom

In [104]:
def trk_custom(patient_folderpath: str, 
               rec_foldername: str = "rec", 
               trk_foldername: str = "trk", 
               trk_min_length: float = 5,
               tract_count: float = 600000,
               silent = False, 
               **kwargs):
    
    
    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    
    if Path(f"{folderpath}/{trk_foldername}/{patientid}_{trk_label}.tt.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid}_{trk_label} already exists. Skipping TRK.")
        return

    path_rec = Path(patient_folderpath) / rec_foldername
    # print(path_rec)
    fib_file = list(path_rec.glob(f'{patientid}.fib.gz'))[0].name
    if not silent:
        print(f"Selected {fib_file} for fiber tracking.")

    roi = kwargs.get("trk_roi", ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior", "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior", "FreeSurferSeg:CC_Anterior"])
    if not silent:
        print(roi)
    !mkdir -p {folderpath}/{trk_foldername}

    cmd = [
        "dsi_studio",
        "--action=trk",
        f"--source={folderpath}/{rec_foldername}/{patientid}.fib.gz",
        f"--seed={','.join(roi)}",
        f"--ter={','.join(roi)},dilation,dilation,negate",
        f"--min_length={trk_min_length}",
        f"--tract_count={tract_count}",
        f"--output={folderpath}/{trk_foldername}/{patientid}_{trk_label}.tt.gz",
        f"--export=stat",
        # " > /dev/null 2>&1"
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Successfully tracked fibers for {patientid}_{trk_label}.")

    return

### TRKconnections

In [ ]:
SHIFT = {"right": "shiftnx", "left": "shiftx"}
CC_REGIONS = ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior",
              "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior",
              "FreeSurferSeg:CC_Anterior"]

def trk_connections(patient_folderpath: str,
               rec_foldername: str = "rec",
               trk_foldername: str = "trk",
               trk_min_length: float = 5,
               tract_count: float = 200000,
               roi_position: str = "left",
               silent=False,
               **kwargs):

    if roi_position not in ("left", "right"):
        raise ValueError("roi_position must be 'left' or 'right'")
    # termination regions go on the side opposite the ROI
    ter_side = "right" if roi_position == "left" else "left"
    sh = SHIFT[ter_side]

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    out_path = folderpath / trk_foldername / f"{patientid}_{trk_label}.tt.gz"

    if out_path.exists():
        if not silent:
            print(f"Reconstruction for {patientid}_{trk_label} already exists. Skipping TRK.")
        return

    path_rec = folderpath / rec_foldername
    fib_file = list(path_rec.glob(f"{patientid}.fib.gz"))[0].name
    if not silent:
        print(f"Selected {fib_file} for fiber tracking.")

    seed = kwargs.get("trk_seed", CC_REGIONS)
    end = kwargs.get("trk_roi")   # e.g. ["HCP-MMP:L_V1_ROI"]; None keeps the old CC behavior
    cc = ",".join(seed)
    if not silent:
        print(seed, end, f"ROI {roi_position}, termination {ter_side}")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    seed_count = kwargs.get("seed_count", tract_count*10)

    cmd = [
        "dsi_studio",
        "--action=trk",
        f"--source={path_rec / fib_file}",
        f"--seed={cc}",
        f"--ter={cc},{sh}",                      # 1. shifted 1 voxel
        f"--ter2={cc},{sh},{sh},dilation",       # 2. shifted 2 voxels, dilated
    ]
    if end:
        cmd.append(f"--end={','.join(end)}")
    cmd += [
        f"--min_length={trk_min_length}",
        f"--tract_count={tract_count}",
        f"--seed_count={seed_count}",
        f"--output={out_path}",
        "--export=stat",
    ]


    loops = 4

    for i in range(loops):
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0 or not out_path.exists():
            print(result.returncode)
            if i == (loops-1):
                print(f"Tracking failed for {patientid}_{trk_label}.")
                if not silent:
                    print(result.stdout[-1000:], result.stderr[-1000:])
                return
        print("repeat")
        

    if not silent:
        print(f"Successfully tracked fibers for {patientid}_{trk_label}.")
    return

## READSTATS

In [106]:
dti_metrics = [
                "number of tracts", "mean length(mm)", "median length(mm)", 
                "span(mm)", "curl", "elongation", "total volume(mm^3)", 
                "LPS quarter volume(mm^3)", "2nd and 3rd quarter volume(mm^3)", 
                "RAI quarter volume(mm^3)", "total surface area(mm^2)", 
                "total radius of end regions(mm)", "total area of end regions(mm^2)",
                "irregularity", "area of LPS end region (mm^2)", "radius of LPS end region (mm)",
                "volume of LPS end branches", "area of RAI end region(mm^2)",
                "radius of RAI end region(mm)", "volume of RAI end branches", 
                "fa", "md", "ad", "rd", "vol", # <---- important stuff maybe idk lol
                "fa of LPS quarter", "md of LPS quarter", "ad of LPS quarter", "rd of LPS quarter",	"vol of LPS quarter", 
                "fa of RAI quarter", "md of RAI quarter", "ad of RAI quarter", "rd of RAI quarter", "vol of RAI quarter",
                ]

In [107]:
found_metrics = []
def readstats(patient_folderpath: str, 
              trk_foldername: str = "trk", 
              silent = False, **kwargs):
    
    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    
    if Path(f"{folderpath}/{trk_foldername}/{patientid}_{trk_label}.tt.gz.stat.txt").exists() == False:
        print("huh trk dont exist?")
        return "TRKNotFound"

    tsv_filepath = Path(patient_folderpath) / trk_foldername / f"{patientid}_{trk_label}.tt.gz.stat.txt"

    tsv_file = f"{patientid}_{trk_label}.tt.gz.stat.txt"
    if not silent:
        print(f"Selected {tsv_file} for reading.")

    df = pd.read_csv(tsv_filepath, sep='\t', header=None, names=["metric", "value"])
    global found_metrics 
    found_metrics = df.iloc[:, 0].tolist()
    # print(df)
    met2val = df.set_index("metric")["value"]
    # print(found_metrics)
    read_metrics = kwargs.get("read_metrics", found_metrics)
    # print(read_metrics)
    if read_metrics == []:
        read_metrics = found_metrics
    
    output = {}
    for i in read_metrics:
        value = met2val[i]
        output[i] = value

    return output

## Compile

In [108]:
def compile(group_folderpath: str, target_foldername: str, output_folderpath: str, silent = False, **kwargs):

    !mkdir -p {Path(output_folderpath)}

    for i in Path(group_folderpath).iterdir():
        if i.is_dir():
            for j in i.iterdir():
                if j.name == target_foldername:
                    for k in j.iterdir():
                        shutil.copy(k, Path(output_folderpath))


    return None

# Pipeline

In [109]:
def chain(inputpath: str, functions: list, silent: bool = False, **kwargs):
    if Path(inputpath).exists() == False:
        print(f" {inputpath} does not exist.")
        return
    
    chain_output = {}

    for func in functions:
        start = time.perf_counter()
        if not silent:
            print(f" \033[1mRunning {func.__name__.upper()} for {Path(inputpath).name}...\033[0m")
        func_output = func(inputpath, silent=True, **kwargs)
        if func_output != None:
            chain_output[func.__name__] = func_output
        end = time.perf_counter()
        # print(func_output)
        if not silent:
            print(f" Finished {func.__name__.upper()} for {Path(inputpath).name}. ({end - start:.6f}s)\n")

    
    return chain_output

In [110]:
def clean_path(path, server_map={"R730XD": "1248"}):
    if f"/mnt/{server_map['R730XD']}/" in path:
        return path
    parts = path.replace("\\", "/").lstrip("/").split("/")
    return Path("/mnt/" + "/".join([server_map[parts[0].upper()]] + parts[1:]))

In [ ]:
# Must have auto skip to avoid redundancy!!!!
def pipeline(groupinputpath: str, groupoutputpath: str, 
                                  functions: list = [bet, eddy, src, rec, trk, readstats], 
                                  silent: bool = False,
                                  collect_filename: str = "data",
                                  **kwargs,
                                  ):

    # Clean
    group_in_path_clean = clean_path(groupinputpath)
    group_out_path_clean = clean_path(groupoutputpath)
    !mkdir -p {group_out_path_clean}

    if Path(group_in_path_clean).exists() == False:
        print(f"{group_in_path_clean} does not exist. Please check again.")
        return

    if not silent:
        print(f"\033[1m\033[4mRunning full batch pipeline...\033[0m\n")

    # Batch D2N
    if not kwargs.get("skip_d2n", False):
        print(f"From:\t{group_in_path_clean}\n To:\t{group_out_path_clean}")
        empty_folders = []
        print("\n\033[4mStarting Batch D2N.\033[0m")
        for d_patient_folder in Path(group_in_path_clean).iterdir():
            if d_patient_folder.is_dir():
                patientid = d_patient_folder.name
                if any(d_patient_folder.iterdir()):
                    if any("DeIdentified" in f.name for f in Path(d_patient_folder).iterdir()):
                        None
                        # print(f"Skipped D2N for patient \033[4m{patientid}\033[0m because deidentified.")
                    else:
                        print(f"Running D2N for patient \033[4m{patientid}\033[0m...")
                        d2ninfo = d2n(d_patient_folder, group_out_path_clean, silent=True)
                        if d2ninfo["d2n_empty_folders"] != []:
                            for i in d2ninfo["d2n_empty_folders"]:
                                empty_folders.append(i)
                else:
                    print(f"Skipped empty folder {patientid}.")
        print(f"Completed batch D2N.")

        if empty_folders != []:
            print(f"Deleting {len(empty_folders)} empty folder(s).")
            for i in empty_folders:
                !rm -R {group_out_path_clean}/{i}
    else:
        print("Skip d2n")
    

    # Batch Chain
    readstats_comp = {}
    lsfunc = [f.__name__ for f in functions]
    print(f"\n\033[4mStarting Batch Chain with {lsfunc}.\033[0m")
    for n_patient_folder in Path(group_out_path_clean).iterdir():
        if n_patient_folder.is_dir():
            
            patientid_date = n_patient_folder.name

            if patientid_date in kwargs.get("blacklist", []):
                print(f"Skipping blacklisted: {patientid_date}\n")
                continue
            
            print(f"Running Chain for patient \033[4m{patientid_date}\033[0m...")
            chaininfo = chain(
                inputpath=n_patient_folder, 
                functions=functions, 
                silent=silent,
                **kwargs,
                )
            sys.stdout.write("\033[F\033[K" * 2)
            if readstats in functions and chaininfo["readstats"] != "TRKNotFound":
                readstats_comp[patientid_date] = chaininfo["readstats"] # {patientid_date : readstats}
            elif readstats in functions and chaininfo["readstats"] == "TRKNotFound":
                print(f"Skipped READSTATS for {patientid_date} because TRK not found.")
            

    print("Completed batch Chain.")
    
    # Collect
    if "readstats" in lsfunc:

        df = pd.DataFrame({
            "PatientID": [],
            "Date": [],
        })


        read_metrics = found_metrics

        for metric in read_metrics:
            df[metric] = None

        for patient in readstats_comp:
            id, date = patient.split("_")

            new_row = {"PatientID": id, "Date": date}
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            mask = (df["PatientID"] == id) & (df["Date"] == date)
            for metric in read_metrics:
                df.loc[mask, metric] = readstats_comp[patient][metric]
    
        print("\n", df)
        
        df.to_csv(f"{group_out_path_clean}/{collect_filename}.csv", index=False)

        return Path(f"{group_out_path_clean}/{collect_filename}.csv")

    else:
        print("READSTATS not included in functions list. Skipping data collection.")
    
    

# Run

## Full Pipeline Run

In [ ]:
# Base
inputpath   : str   = r"\\R730XD\open\dataset5\gLymphVIDA\Ctrl"
outputpath  : str   = r"\\R730XD\open\finley\hydrocephalus\Ctrl"
csv_name    : str   = "QSDR"
functions   : list  = [ # Done in list order
    bet, eddy, src, rec_custom, 
    # trk_custom, readstats,
    # rec_qsdr,
                       ]
silent      : bool  = False

# D2N
skip_d2n    : bool  = True

# REC
rec_method  : int   = 1
rec_param0  : float = 1.25
rec_param1  : float = 2
rec_foldername : str = "rec_qsdr"

# TRK # most likely skip, since trk is batch
trk_label   : str   = "QSDR"
trk_roi     : list  = ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior", "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior", "FreeSurferSeg:CC_Anterior"]
trk_min_length : float = 5
tract_count : float = 600000

# READSTATS
read_metrics: list  = [] # Leave empty to get all metrics


# Other
blacklist  : list  = ["5694483_20250903"]

In [116]:
pipeline(
    # Base
    groupinputpath  = inputpath, 
    groupoutputpath = outputpath,
    functions       = functions, 
    silent          = silent,
    collect_filename= csv_name,

    # D2N
    skip_d2n        = skip_d2n,

    # REC
    rec_method      = rec_method,
    rec_param0      = rec_param0,
    rec_param1      = rec_param1,
    rec_foldername  = rec_foldername,

    # TRK
    trk_roi         = trk_roi, 
    trk_label       = trk_label, 
    trk_min_length  = trk_min_length,
    tract_count     = tract_count,

    # READSTATS
    read_metrics    = read_metrics,

    # Other
    blacklist       = blacklist,
    )

Running full batch pipeline...
 From:	/mnt/1248/open/dataset5/gLymphVIDA/Hydro_1
 To:	/mnt/1248/open/finley/hydrocephalus/Hydro_1
Skip d2n

Starting Batch Chain with ['bet', 'eddy', 'src', 'rec_custom', 'trk_custom', 'readstats'].
Running Chain for patient 6517824_20260603...
 Running BET for 6517824_20260603...
 Finished BET for 6517824_20260603. (0.041569s)

 Running EDDY for 6517824_20260603...
 Finished EDDY for 6517824_20260603. (0.030701s)

 Running SRC for 6517824_20260603...
 Finished SRC for 6517824_20260603. (0.017732s)

 Running REC_CUSTOM for 6517824_20260603...
 Finished REC_CUSTOM for 6517824_20260603. (0.017532s)

 Running TRK_CUSTOM for 6517824_20260603...
 Finished TRK_CUSTOM for 6517824_20260603. (0.018598s)

 Running READSTATS for 6517824_20260603...
 Finished READSTATS for 6517824_20260603. (0.051540s)

Running Chain for patient 8071098_20260323...
 Running BET for 8071098_20260323...
 Finished BET for 8071098_20260323. (0.041729s)

 Running EDDY for 8071098_2026032

KeyboardInterrupt: 

## Batch Variables List

In [112]:
def atlas_format(roi, atlas):
    if atlas == "FreeSurferSeg":
        for i in roi:
            for j in i:
                # print(j)
                if not j.startswith("FreeSurferSeg:"):
                    roi[roi.index(i)][i.index(j)] = "FreeSurferSeg:" + j
    if atlas == "HCP-MMP":
        for i in roi:
            for j in i:
                # print(j)
                if not j.startswith("HCP-MMP:"):
                    roi[roi.index(i)][i.index(j)] = "HCP-MMP:" + j
    return roi

In [113]:
def nest_list(lst):
    newlist = []
    for i in lst:
        newlist.append([i])
    return newlist

In [114]:
# Corpus Callosum Regions
corpus_labels = ["CC_posterior", 
                 "CC_midposterior", 
                 "CC_central", 
                 "CC_midanterior", 
                 "CC_anterior", 
                 ]

corpus_rois = atlas_format(
    nest_list(corpus_labels), 
    atlas = "FreeSurferSeg")

In [ ]:
# Cortical Regions
def add_LR(lst):
    newlist = []
    for i in lst:
        newlist.append("L_" + i)
        newlist.append("R_" + i)
    return newlist

cortical_labels = add_LR([
    "3a", "3b", "1", "2",
    "6mp", "6ma", "SCEF",
    "5m", "5mv", "5L", "4",
    "24dd", "24dv", "23c", "p24pr",
    "6d", "6a", "6v", "6r", "55b",
    "46", "9-46d", "p9-46v", "a9-46v", "9a", "9p", "8C", "8Av", "8Ad", "i6-8", "s6-8",
    "47l", "47s", "47m", "a47r", "p47r", "45",
    "AAIC", "AVI", "MI", "FOP2", "FOP3",
    "PF", "PFt", "PFop", "PFm", "PFcm",
    "TPOJ1", "TPOJ2", "TPOJ3", "STV", "PSL",
    "7AL", "7Am", "7PC", "7PL", "7Pm", "7m", "VIP", "MIP", "LIPd", "LIPv", "AIP",
    "a24pr", "p24pr", "a32pr", "p32pr", "d32", "a24", "p24", "8BM",
    "9m", "10v", "10r", "25", "s32", "p32",
    "OFC", "pOFC", "11l", "13l",
    "SFL", "8BL", "8BM", "9m",
    "7m", "7Pm", "PCV", "POS2",
    "H", "PreS", "EC", "PeEc", "PHA1", "PHA2", "PHA3",
    "TGd", "TGv", "STGa", "TE1a",
    "TE1m", "TE1p", "TE2a", "TE2p", "PHT", "PH", "TF",
    "A4", "A5", "TA2", "STSda", "STSdp",
    "Ig", "PoI1", "PoI2", "PI", "52",
    "V1", "V2", "V3",
    
    # new stuff
    # "4", "3a", "3b", "1", "2", 
    # "6mp", "6ma", "SCEF", "6d", "6v", 
    # "24dd", "24dv", 
    # "5m", "5mv", "5L", "7Am", "7Pm", "7PC", "7m", "VIP", "MIP", 
    # "2v", "PIC", "OP2-3", "OP4", "PFcm", "43", 
    # "46", "9-46d", "a9-46v", "p9-46v", "8C", "8Av", "8BL", "i6-8", "s6-8", "IFJa", "IFJp", "IFSa", "IFSp", "IP1", "IP2", "LIP", "AIP", "7PL", 
    # "EC", "PreS", "H", "PeEc", "PHA1-3", "PHT", "PGp", "POS1", "POS2", "RSC", "v23ab", "d23ab", "31pv", "31pd", "31a", 
    # "10r", "10v", "10d", "9m", "9a", "p32", "a24", "s32", "d32", "25", "47s", "PGi", "PGs", "TPOJ1-3", "STSdp", "STSvp", "TE1a", "TE1m", "TGd", 
    # "44", "45", "47l", "55b", "PSL", "SFL", "STV", "A4", "A5", "TPOJ1"
])

cortical_rois = atlas_format(
    nest_list(cortical_labels), 
    atlas = "HCP-MMP")

## Batch TRK and Readstats

In [48]:
# Batches (Each instance is for a region of interest)
trk_labels = cortical_labels

trk_rois = cortical_rois

# Other
general_label = "QSDR"
blacklist = ["3541363_20260723", "5694483_20250903"]

# Base
inputpath   : str   = r"\\R730XD\open\dataset5\gLymphVIDA\Ctrl"
outputpath  : str   = r"\\R730XD\open\finley\hydrocephalus\Ctrl"
functions   : list  = [trk_custom, readstats]
silent      : bool  = False

# D2N
skip_d2n    : bool  = True

# TRK
trk_min_length : float = 5
tract_count : float = 600000

# REC
rec_foldername : str = "rec_qsdr" # important to set depending on qsdr or dti

# READSTATS
read_metrics: list  = [] # Leave empty to get all metrics

In [49]:
if len(trk_labels) == len(trk_rois):
    print("Proceed")
else:
    raise IndexError("list amount not equal")

# csv_names = [f"data_{label}" for label in trk_labels]

if general_label != "":
    # for i in range(len(trk_labels)):
    #         csv_names[i] = f"{general_label}_{csv_names[i]}"
    g_trk_labels = [None] * len(trk_labels)
    for i in range(len(trk_labels)):
        g_trk_labels[i] = f"{general_label}_{trk_labels[i]}"
    
for i in range(len(trk_labels)):
    # Base
    csv_name    : str   = g_trk_labels[i]
    
    # TRK
    trk_label   : str   = g_trk_labels[i]
    trk_roi     : list  = trk_rois[i]

    print(f"\n\n\033[1m\033[4mRunning pipeline for {trk_label}...\033[0m\n")
    pipeline(
        # Base
        groupinputpath  = inputpath, 
        groupoutputpath = outputpath,
        functions       = functions, 
        silent          = silent,
        collect_filename= csv_name,

        # D2N
        skip_d2n        = skip_d2n,

        # REC
        rec_foldername  = rec_foldername,

        # TRK
        trk_roi         = trk_roi, 
        trk_label       = trk_label, 
        trk_min_length  = trk_min_length,
        tract_count     = tract_count,

        # READSTATS
        read_metrics    = read_metrics,

        # Other
        blacklist       = blacklist,
        )


Proceed


Running pipeline for QSDR_L_4...

Running full batch pipeline...

Skip d2n

Starting Batch Chain with ['trk_custom', 'readstats'].
Running Chain for patient 0029023_20260818...
 Running TRK_CUSTOM for 0029023_20260818...
 Finished TRK_CUSTOM for 0029023_20260818. (0.018888s)

 Running READSTATS for 0029023_20260818...
 Finished READSTATS for 0029023_20260818. (0.023880s)

Running Chain for patient 4457343_20260722...
 Running TRK_CUSTOM for 4457343_20260722...
 Finished TRK_CUSTOM for 4457343_20260722. (0.018162s)

 Running READSTATS for 4457343_20260722...
 Finished READSTATS for 4457343_20260722. (0.023558s)

Running Chain for patient 3598703_20251126...
 Running TRK_CUSTOM for 3598703_20251126...
 Finished TRK_CUSTOM for 3598703_20251126. (0.018439s)

 Running READSTATS for 3598703_20251126...
 Finished READSTATS for 3598703_20251126. (0.024832s)

Running Chain for patient 2328986_20260826...
 Running TRK_CUSTOM for 2328986_20260826...
 Finished TRK_CUSTOM for 2328986_2026

KeyboardInterrupt: 

## Batch TRKconnections and Readstats 

In [131]:
# Batches (Each instance is for a region of interest)
trk_labels = cortical_labels

trk_rois = cortical_rois

# Other
general_label = "QSDRtest_CC_to"
blacklist = ["3541363_20260723", "5694483_20250903"]

# Base
inputpath   : str   = r"\\R730XD\open\dataset5\gLymphVIDA\Hydro_1"
outputpath  : str   = r"\\R730XD\open\finley\hydrocephalus\Hydro_1"
functions   : list  = [trk_connections, readstats]
silent      : bool  = False

# D2N
skip_d2n    : bool  = True

# TRK
trk_min_length : float = 5
tract_count : float = 50000
seed_count     = 5000000

# REC
rec_foldername : str = "rec_qsdr" # important to set depending on qsdr or dti

# READSTATS
read_metrics: list  = [] # Leave empty to get all metrics

In [ ]:
if len(trk_labels) == len(trk_rois):
    print("Proceed")
else:
    raise IndexError("list amount not equal")

# csv_names = [f"data_{label}" for label in trk_labels]

if general_label != "":
    # for i in range(len(trk_labels)):
    #         csv_names[i] = f"{general_label}_{csv_names[i]}"
    g_trk_labels = [None] * len(trk_labels)
    for i in range(len(trk_labels)):
        g_trk_labels[i] = f"{general_label}_{trk_labels[i]}"
    
for i in range(len(trk_labels)):
    # Base
    csv_name    : str   = g_trk_labels[i]
    
    # TRK
    trk_label   : str   = g_trk_labels[i]
    trk_roi     : list  = trk_rois[i]


    #leftrightthingy
    if "L_" in trk_label:
        lr = "left"
    elif "R_" in trk_label:
        lr = "right"
    

    print(f"\n\n\033[1m\033[4mRunning pipeline for {trk_label}...\033[0m\n")
    pipeline(
        # Base
        groupinputpath  = inputpath, 
        groupoutputpath = outputpath,
        functions       = functions, 
        silent          = silent,
        collect_filename= csv_name,

        # D2N
        skip_d2n        = skip_d2n,

        # REC
        rec_foldername  = rec_foldername,

        # TRK
        trk_roi         = trk_roi, 
        trk_label       = trk_label, 
        trk_min_length  = trk_min_length,
        tract_count     = tract_count,
        seed_count     = seed_count,
        roi_position    = lr, # for connection trk


        # READSTATS
        read_metrics    = read_metrics,

        # Other
        blacklist       = blacklist,
        
        )


Proceed


Running pipeline for QSDRtest_CC_to_L_4...

Running full batch pipeline...

Skip d2n

Starting Batch Chain with ['trk_connections', 'readstats'].
Running Chain for patient 6517824_20260603...
 Running TRK_CONNECTIONS for 6517824_20260603...
repeat
repeat
repeat
Tracking failed for 6517824_20260603_QSDRtest_CC_to_L_4.
 Finished TRK_CONNECTIONS for 6517824_20260603. (6.298896s)

 Running READSTATS for 6517824_20260603...
huh trk dont exist?
 Finished READSTATS for 6517824_20260603. (0.006580s)

Skipping readstats for 6517824_20260603 because TRK not found.
Running Chain for patient 8071098_20260323...
 Running TRK_CONNECTIONS for 8071098_20260323...
 Finished TRK_CONNECTIONS for 8071098_20260323. (0.019028s)

 Running READSTATS for 8071098_20260323...
 Finished READSTATS for 8071098_20260323. (0.024615s)

Running Chain for patient 6767874_20260127...
 Running TRK_CONNECTIONS for 6767874_20260127...
 Finished TRK_CONNECTIONS for 6767874_20260127. (0.017704s)

 Running READSTATS 

## Correlation stuff

In [251]:
group_folderpath = "/mnt/1248/open/finley/hydrocephalus/Ctrl/"
target_foldername = "rec_qsdr"
output_folderpath = "/mnt/1248/open/finley/hydrocephalus/pilot/correlational3/ctrl"

compile(group_folderpath, target_foldername, output_folderpath)

In [ ]:
import random

bigpath = "/mnt/1248/open/finley/hydrocephalus/pilot/correlational2/"

id = []
hydro = []

for i in ["ctrl", "hydro"]:

    for j in (Path(bigpath) / i).iterdir():
        relative = str(j).replace("/mnt/1248/", "", 1)
        filepath = r"\\R730XD" + "\\" + relative.replace("/", "\\")
        id.append(filepath)
        if i == "ctrl":
            hydro.append(0)
        else: 
            hydro.append(random.uniform(1.001, 0.999))


    
df = pd.DataFrame({
    "id": id,
    "hydro": hydro
})

print(df)

df.to_csv((Path(bigpath)/"list.csv"), index=False)
df.drop(columns="hydro").to_csv((Path(bigpath)/"list.txt"), index=False, header=False)

# Testing

In [ ]:
SHIFT = {"right": "shiftnx", "left": "shiftx"}
CC_REGIONS = ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior",
              "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior",
              "FreeSurferSeg:CC_Anterior"]

def trk_connections(patient_folderpath: str,
               rec_foldername: str = "rec",
               trk_foldername: str = "trk",
               trk_min_length: float = 5,
               tract_count: float = 200000,
               roi_position: str = "left",
               silent=False,
               **kwargs):

    if roi_position not in ("left", "right"):
        raise ValueError("roi_position must be 'left' or 'right'")
    # termination regions go on the side opposite the ROI
    ter_side = "right" if roi_position == "left" else "left"
    sh = SHIFT[ter_side]

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    out_path = folderpath / trk_foldername / f"{patientid}_{trk_label}.tt.gz"

    if out_path.exists():
        if not silent:
            print(f"Reconstruction for {patientid}_{trk_label} already exists. Skipping TRK.")
        return

    path_rec = folderpath / rec_foldername
    fib_file = list(path_rec.glob(f"{patientid}.fib.gz"))[0].name
    if not silent:
        print(f"Selected {fib_file} for fiber tracking.")

    seed = kwargs.get("trk_seed", CC_REGIONS)
    end = kwargs.get("trk_roi")   # e.g. ["HCP-MMP:L_V1_ROI"]; None keeps the old CC behavior
    cc = ",".join(seed)
    if not silent:
        print(seed, end, f"ROI {roi_position}, termination {ter_side}")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "dsi_studio",
        "--action=trk",
        f"--source={path_rec / fib_file}",
        f"--seed={cc}",
        f"--ter={cc},{sh}",                      # 1. shifted 1 voxel
        f"--ter2={cc},{sh},{sh},dilation",       # 2. shifted 2 voxels, dilated
    ]
    if end:
        cmd.append(f"--end={','.join(end)}")
    cmd += [
        f"--min_length={trk_min_length}",
        f"--tract_count={tract_count}",
        f"--output={out_path}",
        "--export=stat",
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0 or not out_path.exists():
        print(f"Tracking failed for {patientid}_{trk_label}.")
        if not silent:
            print(result.stdout[-1000:], result.stderr[-1000:])
        return

    if not silent:
        print(f"Successfully tracked fibers for {patientid}_{trk_label}.")
    return


def trk_custom_regions(patient_folderpath: str, regions: list,
                       roi_position: str = "left",
                       atlas: str = "HCP-MMP", **kwargs):
    for r in regions:
        trk_custom(patient_folderpath,
                   trk_label=r,
                   trk_roi=[f"{atlas}:{r}"],
                   roi_position=roi_position,
                   **kwargs)

In [ ]:
trk_connections(patient_folderpath = Path("/mnt/1248/open/finley/hydrocephalus/Ctrl/2083150_20251124"),
               rec_foldername = "rec_qsdr",
               trk_foldername = "trk_test",
               trk_min_length = 5,
               tract_count = 200000,
               roi_position = "left",
               silent=False,
               **kwargs):

In [ ]:
trk_connections(patient_folderpath = Path("/mnt/1248/open/finley/hydrocephalus/Ctrl/2083150_20251124"),
           trk_label="L_4",
           trk_roi=["HCP-MMP:L_4"],
           roi_position="left",    # ROI on the left, so termination regions go on the right
           rec_foldername = "rec_qsdr",
           trk_foldername = "trk_test",
           silent=False)

Selected 2083150_20251124.fib.gz for fiber tracking.
['FreeSurferSeg:CC_Posterior', 'FreeSurferSeg:CC_Mid_Posterior', 'FreeSurferSeg:CC_Central', 'FreeSurferSeg:CC_Mid_Anterior', 'FreeSurferSeg:CC_Anterior'] ['HCP-MMP:L_4'] ROI left, termination right
Successfully tracked fibers for 2083150_20251124_L_4.


In [64]:
cortical_rois

[['HCP-MMP:L_4'],
 ['HCP-MMP:R_4'],
 ['HCP-MMP:L_3a'],
 ['HCP-MMP:R_3a'],
 ['HCP-MMP:L_3b'],
 ['HCP-MMP:R_3b'],
 ['HCP-MMP:L_1'],
 ['HCP-MMP:R_1'],
 ['HCP-MMP:L_2'],
 ['HCP-MMP:R_2'],
 ['HCP-MMP:L_6mp'],
 ['HCP-MMP:R_6mp'],
 ['HCP-MMP:L_6ma'],
 ['HCP-MMP:R_6ma'],
 ['HCP-MMP:L_SCEF'],
 ['HCP-MMP:R_SCEF'],
 ['HCP-MMP:L_6d'],
 ['HCP-MMP:R_6d'],
 ['HCP-MMP:L_6v'],
 ['HCP-MMP:R_6v'],
 ['HCP-MMP:L_24dd'],
 ['HCP-MMP:R_24dd'],
 ['HCP-MMP:L_24dv'],
 ['HCP-MMP:R_24dv'],
 ['HCP-MMP:L_5m'],
 ['HCP-MMP:R_5m'],
 ['HCP-MMP:L_5mv'],
 ['HCP-MMP:R_5mv'],
 ['HCP-MMP:L_5L'],
 ['HCP-MMP:R_5L'],
 ['HCP-MMP:L_7Am'],
 ['HCP-MMP:R_7Am'],
 ['HCP-MMP:L_7Pm'],
 ['HCP-MMP:R_7Pm'],
 ['HCP-MMP:L_7PC'],
 ['HCP-MMP:R_7PC'],
 ['HCP-MMP:L_7m'],
 ['HCP-MMP:R_7m'],
 ['HCP-MMP:L_VIP'],
 ['HCP-MMP:R_VIP'],
 ['HCP-MMP:L_MIP'],
 ['HCP-MMP:R_MIP'],
 ['HCP-MMP:L_2v'],
 ['HCP-MMP:R_2v'],
 ['HCP-MMP:L_PIC'],
 ['HCP-MMP:R_PIC'],
 ['HCP-MMP:L_OP2-3'],
 ['HCP-MMP:R_OP2-3'],
 ['HCP-MMP:L_OP4'],
 ['HCP-MMP:R_OP4'],
 ['HCP-MMP:L_PFcm'